# FounderOS — AMD GPU Compute Demonstration

**AMD AI Developer Hackathon — Track 3 (Unicorn/Open Innovation)**

This notebook demonstrates **direct AMD GPU compute usage** on the provided AMD cloud instance with ROCm 7.2, PyTorch 2.9, and the HuggingFace Transformers library.

## What This Shows
1. **AMD GPU Detection** — Verifying ROCm and GPU availability
2. **GPU Compute Benchmark** — Matrix multiplication GFLOPS on AMD GPU
3. **LLM Inference on AMD GPU** — Loading and running a model directly on the AMD GPU via HuggingFace Transformers
4. **Performance Metrics** — Token throughput, latency, and memory utilization
5. **FounderOS Integration** — How the local GPU connects to the multi-agent app

In [ ]:
# ============================================================
# Step 1: Verify AMD GPU and ROCm Environment
# ============================================================
import subprocess

print("=" * 60)
print("  AMD GPU Environment Verification")
print("=" * 60)

# Check ROCm version
try:
    result = subprocess.run(["rocm-smi", "--showproductname"], capture_output=True, text=True)
    print("\n[ROCm GPU Info]")
    print(result.stdout)
except FileNotFoundError:
    print("rocm-smi not found, trying alternative...")

# Check PyTorch ROCm support
import torch
print(f"\n[PyTorch]")
print(f"  Version: {torch.__version__}")
print(f"  ROCm Build: {torch.version.hip}")
print(f"  CUDA Available (via ROCm): {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"  GPU Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}")
        print(f"    Memory: {props.total_memory / 1e9:.1f} GB")
        print(f"    Compute: {props.major}.{props.minor}")
        print(f"    Multi-Processor Count: {props.multi_processor_count}")
else:
    print("  WARNING: No GPU detected via PyTorch")
    print("  Trying CPU-only mode...")

In [ ]:
# ============================================================
# Step 2: GPU Memory and Compute Benchmark
# ============================================================
import torch
import time

device = "cuda" if torch.cuda.is_available() else "cpu"

if torch.cuda.is_available():
    print("[GPU Memory Status]")
    for i in range(torch.cuda.device_count()):
        total = torch.cuda.get_device_properties(i).total_memory / 1e9
        allocated = torch.cuda.memory_allocated(i) / 1e9
        cached = torch.cuda.memory_reserved(i) / 1e9
        print(f"  GPU {i}: {allocated:.2f} GB allocated / {cached:.2f} GB cached / {total:.1f} GB total")

# Matrix multiply benchmark — proves GPU compute works
print("\n[GPU Compute Test: 4096x4096 Matrix Multiply]")
x = torch.randn(4096, 4096, device=device)
y = torch.randn(4096, 4096, device=device)

# Warmup
for _ in range(3):
    _ = torch.matmul(x, y)
if torch.cuda.is_available():
    torch.cuda.synchronize()

# Timed run
start = time.time()
z = torch.matmul(x, y)
if torch.cuda.is_available():
    torch.cuda.synchronize()
elapsed = time.time() - start

gflops = (2 * 4096**3) / elapsed / 1e9
print(f"  Time: {elapsed*1000:.1f} ms")
print(f"  Estimated GFLOPS: {gflops:.1f}")
print(f"  Device: {device}")

del x, y, z
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Step 3: LLM Inference on AMD GPU

We load a HuggingFace model directly onto the AMD GPU and run inference. This is the **core AMD compute demonstration** — the LLM runs entirely on the local AMD GPU, no external API calls.

We use `transformers` with `torch.float16` for efficient GPU memory usage.

In [ ]:
# ============================================================
# Step 3a: Load a model onto the AMD GPU
# ============================================================
# We use a small but capable model that downloads quickly.
# "google/flan-t5-small" (~250MB) downloads fast and proves the GPU pipeline.
# If you want a larger model, try: "microsoft/phi-2" (~2.7GB)

import os
import time

GPU_MODEL = os.environ.get("GPU_MODEL", "google/flan-t5-small")

print(f"[Loading model onto AMD GPU: {GPU_MODEL}]")
print(f"  Device: {device}")
print(f"  This may take 1-2 minutes for first-time download...")
print()

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

load_start = time.time()

tokenizer = AutoTokenizer.from_pretrained(GPU_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(
    GPU_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

if not torch.cuda.is_available():
    model = model.to(device)

load_time = time.time() - load_start
print(f"  Model loaded in {load_time:.1f}s")
print(f"  Device: {model.device}")
print(f"  Dtype: {model.dtype}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

# Show GPU memory after loading
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU Memory Used: {allocated:.2f} GB / {total:.1f} GB ({allocated/total*100:.0f}%)")

In [ ]:
# ============================================================
# Step 3b: Run inference — FounderOS Strategist Agent simulation
# ============================================================
import time

prompt = "I'm building an AI-powered meal planning app for busy professionals. What are the 3 most important things to validate first?"

print("=" * 60)
print("  AMD GPU Inference — FounderOS Strategist Agent")
print("=" * 60)
print(f"\n[Prompt]\n{prompt}")
print(f"\n[Model] {GPU_MODEL}")
print(f"[Device] {model.device}")

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Warmup run
_ = model.generate(**inputs, max_new_tokens=10)

# Timed inference
start = time.time()
output = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7)
if torch.cuda.is_available():
    torch.cuda.synchronize()
elapsed = time.time() - start

response = tokenizer.decode(output[0], skip_special_tokens=True)

# Count output tokens
output_tokens = len(tokenizer.encode(response)) - len(tokenizer.encode(prompt))

print(f"\n[Response] ({elapsed*1000:.0f}ms, ~{output_tokens} tokens)")
print("-" * 50)
print(response)
print("-" * 50)

print(f"\n[Performance Metrics]")
print(f"  Inference time: {elapsed*1000:.0f}ms")
print(f"  Output tokens: ~{output_tokens}")
print(f"  Throughput: {output_tokens/max(elapsed, 0.001):.1f} tokens/second")
print(f"  Memory after inference: {torch.cuda.memory_allocated(0)/1e9:.2f} GB" if torch.cuda.is_available() else "")

In [ ]:
# ============================================================
# Step 4: Multi-Agent Inference Benchmark
# ============================================================
# Simulate all 6 FounderOS agents running on the AMD GPU

agent_queries = [
    ("Strategist", "What are the key steps to validate a SaaS startup idea in 30 days?"),
    ("Analyst", "Analyze the TAM for an AI-powered developer tools startup."),
    ("Researcher", "What are the latest trends in AI agent frameworks for 2025?"),
    ("Writer", "Write a compelling one-sentence pitch for a meal planning AI app."),
    ("Coder", "Suggest the best Python web framework for an AI agent backend."),
    ("Coach", "I have 8 hours before a hackathon deadline. What should I focus on?"),
]

print("[Multi-Agent AMD GPU Inference Benchmark]")
print("=" * 60)
print(f"{'Agent':<14} {'Tokens':>6} {'Time(ms)':>10} {'Tok/sec':>9}")
print("-" * 60)

total_tokens = 0
total_time = 0

for agent_name, query in agent_queries:
    inputs = tokenizer(query, return_tensors="pt").to(model.device)
    
    start = time.time()
    output = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.time() - start
    
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    out_tokens = len(tokenizer.encode(response)) - len(tokenizer.encode(query))
    throughput = out_tokens / max(elapsed, 0.001)
    
    print(f"{agent_name:<14} {out_tokens:>6} {elapsed*1000:>10.0f} {throughput:>9.1f}")
    
    total_tokens += out_tokens
    total_time += elapsed

print("-" * 60)
print(f"{'TOTAL':<14} {total_tokens:>6} {total_time*1000:>10.0f} {total_tokens/max(total_time,0.001):>9.1f}")
print(f"\n  All 6 FounderOS agents ran successfully on the AMD GPU!")
print(f"  Avg throughput: {total_tokens/max(total_time,0.001):.1f} tokens/second")

In [ ]:
# ============================================================
# Step 5: GPU Utilization Summary
# ============================================================
import torch

print("[GPU Utilization After All Inference]")
print("-" * 40)

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        total = torch.cuda.get_device_properties(i).total_memory / 1e9
        allocated = torch.cuda.memory_allocated(i) / 1e9
        cached = torch.cuda.memory_reserved(i) / 1e9
        utilization = (cached / total) * 100
        print(f"  GPU {i}:")
        print(f"    Memory Used: {allocated:.2f} GB / {total:.1f} GB")
        print(f"    Memory Reserved: {cached:.2f} GB ({utilization:.1f}%)")

# Also try rocm-smi for live utilization
try:
    result = subprocess.run(["rocm-smi"], capture_output=True, text=True)
    print("\n[rocm-smi output]")
    print(result.stdout[:1000])
except FileNotFoundError:
    print("\n  rocm-smi not available")

## Step 6: FounderOS Integration

The model running on the AMD GPU can be connected to FounderOS by setting:
- `USE_LOCAL_GPU=true` in the `.env` file
- `LOCAL_VLLM_URL=http://localhost:8080/v1` (if using vLLM) or wrapping the transformers model in an OpenAI-compatible API

This routes all 6 agent inferences through the local AMD GPU instead of external APIs.

In [ ]:
# ============================================================
# Step 6: FounderOS Integration Summary
# ============================================================
print("[FounderOS + AMD GPU Integration]")
print("=" * 50)
print(f"\n  Model: {GPU_MODEL}")
print(f"  Device: {model.device}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"\n  To connect to FounderOS backend:")
print(f"    1. Set USE_LOCAL_GPU=true in .env")
print(f"    2. Set LOCAL_VLLM_URL=http://localhost:8080/v1")
print(f"    3. Start vLLM with: python -m vllm.entrypoints.openai.api_server --model {GPU_MODEL}")
print(f"    4. Or wrap this transformers model in a FastAPI OpenAI-compatible endpoint")
print(f"\n  FounderOS will route all 6 agent queries through the AMD GPU.")
print(f"\n  Architecture:")
print(f"    User -> FounderOS Frontend -> Backend (FastAPI)")
print(f"      -> Model Router -> AMD GPU (vLLM or transformers)")
print(f"      -> Agent Graph (LangGraph) -> Response")

## Summary

This notebook demonstrates:

1. **AMD ROCm Detection** — Confirmed GPU, PyTorch ROCm build, memory
2. **GPU Compute Benchmark** — Matrix multiplication GFLOPS
3. **LLM Inference on AMD GPU** — Loaded and ran a model locally via HuggingFace Transformers
4. **Multi-Agent Benchmark** — All 6 FounderOS agents simulated on AMD GPU
5. **GPU Utilization** — Memory and compute metrics
6. **FounderOS Integration** — Connection path to the app backend

### AMD Compute Usage Summary
- **Hardware**: AMD GPU (gfx1100) via ROCm 7.2
- **Framework**: PyTorch 2.9 (ROCm-enabled)
- **Model**: HuggingFace Transformers with float16
- **Use Cases**: LLM inference for all 6 FounderOS agents
- **Integration**: OpenAI-compatible API via vLLM, or direct transformers inference